# Notebook 04 — Hypothesis Testing

| Field | Detail |
|---------|----------|
| Member | Member D — Hypothesis Analyst |
| Project | moby/moby (Docker Engine) |
| Layer | Week 13 — Hypothesis Testing |

---

## Research Questions

### RQ1
Apakah probabilitas Pull Request (PR) pada repository `moby/moby` berhasil di-merge berbeda secara signifikan dari 75%?

### RQ2
Apakah rata-rata jumlah issue yang dibuat per minggu berbeda secara signifikan antara tahun 2025 dan 2026?

---

### Data yang Digunakan

Notebook ini menggunakan:

- `pull_requests_clean.csv`
- `issues_clean.csv`

yang telah dipersiapkan oleh Member A (Data Engineer).  
Notebook ini juga menggunakan hasil estimasi dari Notebook 02 (Member B) dan hasil confidence interval dari Notebook 03 (Member C).

## AI Disclosure

**Member:** Malik Alfat Muzaki — Hypothesis Testing Analyst

| Task | Tool | Prompt Summary | Pemanfaatan Output |
|------|------|----------------|--------------------|
| Pengecekan apakah kode sudah benar | Gemini | "cek apakah kode tersebut sudah benar?" | Digunakan sebagai validasi kode dan membantu mengidentifikasi kesalahan implementasi. |
| Pengecekan logika yang salah | Gemini | "mengapa bagian tersebut salah/kurang tepat" | Digunakan untuk memahami kesalahan logika serta memperoleh masukan untuk perbaikan analisis dan implementasi. |

## 1. Persiapan Environment

Library yang digunakan pada pengujian hipotesis ini adalah **Pandas**, **NumPy**, dan **SciPy**.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

## 2. RQ1 — Probabilitas Pull Request Berhasil di-Merge

Setiap Pull Request memiliki dua kemungkinan hasil:

- Berhasil di-merge (success)
- Tidak di-merge (failure)

Karena hanya memiliki dua kemungkinan hasil, maka data dapat dimodelkan menggunakan **distribusi Bernoulli**.

Tujuan pengujian adalah menentukan apakah probabilitas merge PR berbeda secara signifikan dari **75%**.

### Hipotesis

Hipotesis nol:

$$H_0 : p = 0.75$$

Hipotesis alternatif:

$$H_1 : p \neq 0.75$$

dengan:

- $p$ = probabilitas sebuah Pull Request berhasil di-merge.

**Metode Uji:** One-Sample Proportion Z-Test (dua sisi)

**Tingkat signifikansi:** $\alpha = 0.05$

In [ ]:
# ==========================
# LOAD DATA
# ==========================

pr_df = pd.read_csv("../data/clean/pull_requests_clean.csv")

# Hanya PR yang sudah memiliki hasil final
pr_df = pr_df[pr_df["state"] == "closed"]

# ==========================
# SAMPLE STATISTICS
# ==========================

n = len(pr_df)

merged = pr_df["is_merged"].sum()

p_hat = merged / n

# ==========================
# HYPOTHESIS
# ==========================

p0 = 0.75

# ==========================
# ONE SAMPLE PROPORTION Z TEST
# ==========================

se = np.sqrt(
    p0 * (1 - p0) / n
)

z = (p_hat - p0) / se

p_value = 2 * (1 - norm.cdf(abs(z)))

alpha = 0.05

if p_value < alpha:
    decision = "Reject H0"
else:
    decision = "Fail to Reject H0"

# ==========================
# OUTPUT
# ==========================

print(f"Jumlah Sampel (n)                : {n}")
print(f"Jumlah PR Merge                  : {merged}")
print(f"Estimasi Probabilitas Merge (p̂)  : {p_hat:.4f}")
print(f"Nilai H0 (p0)                    : {p0}")
print(f"Standard Error                   : {se:.6f}")
print(f"Z Statistic                      : {z:.4f}")
print(f"p-value                          : {p_value:.6f}")
print(f"Keputusan                        : {decision}")

### Interpretasi Hasil — RQ1

Hasil pengujian menunjukkan:

- Jumlah sampel (n) = **884**
- Jumlah PR berhasil di-merge = **743**
- Estimasi probabilitas merge:

$$\hat{p} = 0.8405$$

- Statistik uji:

$$Z = 6.2139$$

- p-value $< 0.001$

Karena **p-value lebih kecil dari** $\alpha = 0.05$, maka **hipotesis nol ditolak**.

Dengan demikian terdapat **bukti statistik yang kuat** bahwa probabilitas Pull Request berhasil di-merge **berbeda secara signifikan dari 75%**.

Nilai estimasi probabilitas merge sebesar **84.05%** menunjukkan bahwa tingkat keberhasilan merge PR pada repository `moby/moby` **relatif tinggi** — lebih tinggi dari nilai hipotesis 75%.

> **Catatan:** Hasil ini konsisten dengan Confidence Interval 95% dari Notebook 03, yaitu $[0.8164,\; 0.8646]$, yang tidak mencakup nilai 0.75.

## 3. RQ2 — Perbandingan Laju Pembukaan Issue Tahun 2025 dan 2026

Issue pada repository dapat dipandang sebagai suatu **proses pencacahan (count process)**.

Untuk mengetahui apakah terdapat perubahan aktivitas pelaporan issue, data dibagi menjadi dua kelompok:

- Issue yang dibuat pada **tahun 2025**
- Issue yang dibuat pada **tahun 2026**

Kemudian dilakukan pengujian terhadap **rata-rata jumlah issue yang dibuat setiap minggu**.

> **Catatan Dataset:** Dataset ini mencakup rentang Desember 2025 hingga Mei 2026, sehingga RQ2 membandingkan dua periode yang memang tersedia dalam data.

### Hipotesis

Hipotesis nol:

$$H_0 : \lambda_{2025} = \lambda_{2026}$$

Hipotesis alternatif:

$$H_1 : \lambda_{2025} \neq \lambda_{2026}$$

dengan:

- $\lambda_{2025}$ = rata-rata jumlah issue per minggu tahun 2025
- $\lambda_{2026}$ = rata-rata jumlah issue per minggu tahun 2026

**Metode Uji:** Two-Sample Z-Test (dua sisi)

**Tingkat signifikansi:** $\alpha = 0.05$

In [ ]:
# ==========================
# LOAD DATA
# ==========================

issues_df = pd.read_csv(
    "../data/clean/issues_clean.csv",
    parse_dates=["created_at"]
)

# ==========================
# SPLIT DATA
# ==========================

before = issues_df[
    issues_df["created_at"].dt.year == 2025
].copy()

after = issues_df[
    issues_df["created_at"].dt.year == 2026
].copy()

# ==========================
# ISSUE COUNT PER WEEK
# ==========================

before["week"] = before["created_at"].dt.to_period("W")
after["week"] = after["created_at"].dt.to_period("W")

before_counts = before.groupby("week").size()
after_counts = after.groupby("week").size()

# ==========================
# SAMPLE STATISTICS
# ==========================

x_bar1 = before_counts.mean()
x_bar2 = after_counts.mean()

sigma1 = before_counts.std(ddof=1)
sigma2 = after_counts.std(ddof=1)

n1 = len(before_counts)
n2 = len(after_counts)

# ==========================
# TWO SAMPLE Z TEST
# ==========================

z = (x_bar1 - x_bar2) / np.sqrt(
    (sigma1**2 / n1) +
    (sigma2**2 / n2)
)

p_value = 2 * (
    1 - norm.cdf(abs(z))
)

alpha = 0.05

if p_value < alpha:
    decision = "Reject H0"
else:
    decision = "Fail to Reject H0"

# ==========================
# OUTPUT
# ==========================

print("H0 : λ2025 = λ2026")
print("H1 : λ2025 ≠ λ2026")
print()
print(f"Rata-rata issue/minggu 2025  : {x_bar1:.2f}")
print(f"Rata-rata issue/minggu 2026  : {x_bar2:.2f}")
print(f"Std Dev 2025 (sigma1)        : {sigma1:.4f}")
print(f"Std Dev 2026 (sigma2)        : {sigma2:.4f}")
print(f"Jumlah Minggu 2025 (n1)      : {n1}")
print(f"Jumlah Minggu 2026 (n2)      : {n2}")
print(f"Z Statistic                  : {z:.4f}")
print(f"p-value                      : {p_value:.6f}")
print(f"Keputusan                    : {decision}")

### Interpretasi Hasil — RQ2

Hasil pengujian menunjukkan:

- Rata-rata issue per minggu tahun 2025 = **9.25**
- Rata-rata issue per minggu tahun 2026 = **6.62**

Statistik uji yang diperoleh adalah:

$$Z = 1.0119$$

dengan:

$$p\text{-value} = 0.3116$$

Karena **p-value lebih besar dari** $\alpha = 0.05$, maka **hipotesis nol gagal ditolak**.

Dengan demikian **tidak terdapat bukti statistik yang cukup** untuk menyatakan bahwa rata-rata jumlah issue yang dibuat per minggu pada tahun 2025 berbeda secara signifikan dengan tahun 2026.

Meskipun secara deskriptif rata-rata tahun 2025 terlihat lebih tinggi dibandingkan tahun 2026, perbedaan tersebut **tidak signifikan secara statistik**.

> **Catatan Keterbatasan:** Ukuran sampel tahun 2025 sangat kecil ($n_{2025} = 4$ minggu), sehingga kesimpulan untuk periode 2025 bersifat **terbatas**. Data tahun 2025 yang tersedia hanya mencakup akhir Desember 2025, jauh lebih sedikit dibandingkan tahun 2026 ($n_{2026} = 21$ minggu). Hasil ini perlu diinterpretasikan dengan hati-hati.

## 4. Kesimpulan

| Research Question | Metode Uji | p-value | Keputusan |
|------------------|------------|---------|----------|
| RQ1: Probabilitas PR di-merge ≠ 75%? | One-Sample Proportion Z-Test | < 0.001 | **Tolak H₀** |
| RQ2: Laju issue 2025 ≠ laju issue 2026? | Two-Sample Z-Test | 0.3116 | **Gagal Menolak H₀** |

### Kesimpulan Akhir

Berdasarkan hasil pengujian hipotesis:

**RQ1:** Probabilitas Pull Request berhasil di-merge pada repository `moby/moby` **berbeda secara signifikan dari 75%**, dengan estimasi probabilitas merge sebesar **84.05%** (lebih tinggi dari nilai hipotesis). Bukti statistik sangat kuat dengan $Z = 6.2139$ dan $p\text{-value} < 0.001$.

**RQ2:** Tidak ditemukan bukti statistik yang cukup untuk menyatakan bahwa rata-rata jumlah issue yang dibuat per minggu **berbeda secara signifikan antara tahun 2025 dan 2026**. Meskipun mean 2025 (9.25 issue/minggu) lebih tinggi dari mean 2026 (6.62 issue/minggu), perbedaan ini tidak mencapai taraf signifikansi $\alpha = 0.05$.

Hasil ini menunjukkan bahwa tingkat keberhasilan merge Pull Request **relatif tinggi dan stabil**, sedangkan aktivitas pelaporan issue pada kedua periode masih berada pada tingkat yang secara statistik **serupa**.